<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week7/Day1/defiipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Défi quotidien : Analyse textuelle de livres à l'aide d'un nuage de mots

In [ ]:
import os
import re
import requests
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk import pos_tag, ne_chunk

import spacy
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# =====================================================================
# CONFIGURATION ET CHARGEMENT SÉCURISÉ DES MODÈLES LINGUISTIQUES
# =====================================================================
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('maxent_ne_chunker', quiet=True)
nltk.download('maxent_ne_chunker_tab', quiet=True)
nltk.download('maxent_ne_chunker_numeric', quiet=True)
nltk.download('words', quiet=True)

nlp = spacy.load("en_core_web_sm")
nlp.max_length = 2000000

# URLs officielles d'accès direct aux textes bruts (.txt) du Projet Gutenberg
URLS = [
    "https://gutenberg.org",       # Alice's Adventures in Wonderland
    "https://gutenberg.org",     # Through the Looking-Glass
    "https://gutenberg.org"   # A Tangled Tale
]
BOOK_NAMES = ["Alice in Wonderland", "Through the Looking-Glass", "A Tangled Tale"]

# =====================================================================
# PARTIE 1 : PRÉTRAITEMENT DU TEXTE (EXERCICES 1 À 9)
# =====================================================================
print("--- Étape 1 & 2 : Chargement, Découpage et Nettoyage Regex ---")

def load_texts(urls):
    corpus = []
    for i, url in enumerate(urls):
        response = requests.get(url)
        text = response.text

        # Nettoyage préventif du code HTML/XML parasite
        text = re.sub(r'<[^>]+>', '', text)

        # Découpage chirurgical pour ignorer les mentions légales Gutenberg au début et à la fin
        start_marker = "*** START OF THE PROJECT GUTENBERG"
        end_marker = "*** END OF THE PROJECT GUTENBERG"

        start_idx = text.upper().find(start_marker)
        if start_idx == -1: start_idx = text.upper().find("START")
        end_idx = text.upper().find(end_marker)
        if end_idx == -1: end_idx = text.upper().find("END")

        if start_idx != -1 and end_idx != -1:
            text = text[start_idx:end_idx]

        # Nettoyage Regex : Ne conserve que les caractères alphabétiques et les espaces
        text_cleaned = re.sub(r'[^a-zA-Z\s]', '', text)
        text_cleaned = re.sub(r'\s+', ' ', text_cleaned).strip()
        corpus.append(text_cleaned)
        print(f" • '{BOOK_NAMES[i]}' chargé et élagué ({len(text_cleaned)} caractères).")
    return corpus

corpus_raw = load_texts(URLS)

print("\n--- Étape 2 : Aperçu des 200 premiers caractères de chaque livre ---")
for i, text in enumerate(corpus_raw):
    print(f"\n[{BOOK_NAMES[i]}] : {text[:200]}...")

print("\n--- Étape 3 : Tokenisation (Impression des 150 premiers jetons) ---")
tokenized_books = [word_tokenize(text.lower()) for text in corpus_raw]
for i, tokens in enumerate(tokenized_books):
    print(f" • '{BOOK_NAMES[i]}' (150 premiers tokens) : {tokens[:150]}")

print("\n--- Étape 4 : Suppression des mots vides (NLTK Stopwords) ---")
stop_words = set(stopwords.words('english'))
# Ajout préventif de termes d'en-tête pour isoler le vocabulaire 100% littéraire
stop_words.update(['gutenberg', 'project', 'license', 'ebook', 'online', 'terms', 'shall', 'said'])

filtered_books = [[t for t in tokens if t not in stop_words and len(t) > 2] for tokens in tokenized_books]
print(f" • Vérification du mot vide 'me' après filtrage : {any('me' in book for book in filtered_books)}")

print("\n--- Étape 5 : Racinisation via PorterStemmer (50 premiers) ---")
stemmer = PorterStemmer()
stemmed_books = [[stemmer.stem(t) for t in tokens] for tokens in filtered_books]
print(f" • Livre 1 (Racinisé) : {stemmed_books[0][:50]}")

print("\n--- Étape 6 : Lemmatisation via spaCy en_core_web_sm (50 premiers) ---")
lemmatized_books_str = []
# Désactivation des composants lourds pour traiter les textes complets à haute performance
with nlp.select_pipes(enable=["tok2vec", "tagger", "attribute_ruler", "lemmatizer"]):
    for text in corpus_raw:
        doc = nlp(text)
        lemmas = [token.lemma_.lower() for token in doc if not token.is_stop and token.is_alpha and token.lemma_.lower() not in stop_words and len(token.text) > 2]
        lemmatized_books_str.append(" ".join(lemmas))
print(f" • Livre 1 (Lemmatisé) : {lemmatized_books_str[0].split()[:50]}")

print("\n--- Étape 7 : Analyse comparative Racinisation vs Lemmatisation ---")
print(" > La Racinisation (Stemming) coupe les fins de mots de manière algorithmique brute (ex: 'beautiful' -> 'beauti').")
print(" > La Lemmatisation s'appuie sur un dictionnaire linguistique complet pour renvoyer le mot à sa forme canonique du dictionnaire (ex: 'was' -> 'be').")

print("\n--- Étape 8 & 9 : Étiquettes POS et Entités Nommées (NLTK sur Échantillon) ---")
sample_tags = pos_tag(filtered_books[0][:15])
print(f" • Étiquettes POS : {sample_tags}")
print(f" • Arbre d'Entités Nommées : {ne_chunk(sample_tags)}")

# =====================================================================
# PARTIE 2 : ANALYSE DU TEXTE & NUAGES DE MOTS
# =====================================================================
print("\n--- Étape 10 : Génération des nuages de mots (WordClouds) ---")
os.makedirs("cache_plots", exist_ok=True)

for i, text_clean in enumerate(lemmatized_books_str):
    wordcloud = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(text_clean)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(f"Word Cloud - {BOOK_NAMES[i]}", fontsize=14, pad=10)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(f"cache_plots/wordcloud_book_{i}.png")
    plt.close()
print(" ✅ Les 3 graphiques de nuages de mots ont été sauvegardés dans 'cache_plots/'.")

# =====================================================================
# PARTIE 3 : SAC DE MOTS (BAG OF WORDS - BoW)
# =====================================================================
print("\n--- Étape 11 : Analyse Bag-of-Words (BoW) ---")
# INDICE : Le meilleur texte source est le texte LEMMATISÉ pour éviter la redondance des déclinaisons grammaticales.

vectorizer_bow = CountVectorizer(max_features=5)
bow_matrix = vectorizer_bow.fit_transform(lemmatized_books_str)
bow_words = vectorizer_bow.get_feature_names_out()
bow_counts = bow_matrix.toarray()

print(f"Mots les plus fréquents détectés (BoW) : {list(bow_words)}")
print("\nCoordonnées de la matrice BoW [Document_ID, Word_Index] : Fréquence")
for doc_idx in range(len(BOOK_NAMES)):
    for word_idx, count in enumerate(bow_counts[doc_idx]):
        print(f" • Document: {doc_idx} ('{BOOK_NAMES[doc_idx]}') | Index du mot: {word_idx} ('{bow_words[word_idx]}') -> Trouvé: {count} fois.")

# Traçage des diagrammes circulaires pour l'approche Bag-of-Words
for i in range(len(BOOK_NAMES)):
    plt.figure(figsize=(5, 5))
    plt.pie(bow_counts[i], labels=bow_words, autopct='%1.1f%%', startangle=140)
    plt.title(f"Top 5 Words (BoW) - {BOOK_NAMES[i]}", fontsize=12, pad=10)
    plt.tight_layout()
    plt.savefig(f"cache_plots/pie_chart_bow_book_{i}.png")
    plt.close()

# =====================================================================
# PARTIE 4 : RÉSOLUTION DU PROBLÈME DE FRÉQUENCE PAR TF-IDF
# =====================================================================
print("\n--- Étape 12 : Résolution du problème de fréquence via TF-IDF ---")

# Utilisation des paramètres exigés par la consigne (min_df=1, max_df=2)
# max_df=2 permet d'éliminer automatiquement les mots trop communs présents dans les 3 documents (ex: 'alice', 'think', 'go')
tfidf_vectorizer = TfidfVectorizer(min_df=1, max_df=2)
tfidf_matrix = tfidf_vectorizer.fit_transform(lemmatized_books_str)
tfidf_words = tfidf_vectorizer.get_feature_names_out()
tfidf_scores = tfidf_matrix.toarray()

for i in range(len(BOOK_NAMES)):
    top5_indices = np.argsort(tfidf_scores[i])[-5:]
    top5_words = [tfidf_words[idx] for idx in top5_indices]
    top5_values = [tfidf_scores[i][idx] for idx in top5_indices]

    plt.figure(figsize=(5, 5))
    plt.pie(top5_values, labels=top5_words, autopct='%1.1f%%', startangle=140)
    plt.title(f"Top 5 Discriminant Words (TF-IDF) - {BOOK_NAMES[i]}", fontsize=12, pad=10)
    plt.tight_layout()
    plt.savefig(f"cache_plots/pie_chart_tfidf_book_{i}.png")
    plt.close()
    print(f" • Mots hautement informatifs (TF-IDF) pour '{BOOK_NAMES[i]}' : {top5_words}")

print("\n🎉 Défi terminé avec succès ! Tous les graphiques circulaires et nuages de mots sont disponibles dans le dossier 'cache_plots/'.")


- Analyse critique du BoW classique : Dans l'approche Bag-of-Words, les mots qui dominent sont des termes récurrents comme alice, go ou think. Ces mots ne sont pas informatifs car ils décrivent des actions de dialogue universelles communes à toutes les œuvres, masquant l'intrigue unique de chaque livre.

- Correction apportée par TF-IDF : En appliquant un filtre max_df=2, TF-IDF pénalise mathématiquement les mots présents dans l'intégralité du corpus. Cela permet de faire émerger des termes exclusifs et discriminants propres à chaque intrigue (noms de personnages secondaires ou objets spécifiques), rendant l'analyse contextuelle beaucoup plus fine.